**NB** : **Pour le retard (stunting)**

# Librairies

### Lecture et visualisation des données

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Separation et validation

In [2]:
from sklearn.model_selection import (
    train_test_split, # separer train/test une seule fois
    StratifiedKFold, # validation croisee stratifiee
    cross_validate, # validation croisee avec plusieurs metriques + temps
    GridSearchCV, # recherche exhaustive des meilleurs hyperparametres
)

### Preparation des donnees

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder# Pour encodage
from sklearn.compose import ColumnTransformer
from sklearn.base import clone # Verification apres encodage

### Evaluation

In [4]:
'''
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    fbeta_score,
    make_scorer,
    precision_score,
)
from sklearn.inspection import permutation_importance
'''

'\nfrom sklearn.metrics import (\n    balanced_accuracy_score,\n    classification_report,\n    confusion_matrix,\n    ConfusionMatrixDisplay,\n    fbeta_score,\n    make_scorer,\n    precision_score,\n)\nfrom sklearn.inspection import permutation_importance\n'

### Modele choisie

In [5]:
# A confirmer

In [6]:
# Fixer le random_state
RANDOM_STATE = 42

In [7]:
df = pd.read_parquet('../data/clean/model_stunting.parquet')

print(f"Taille ; {df.shape}")

Taille ; (4864, 15)


In [8]:
df.head()

,age_mois,region,edu_mere,electricite,cuisiniere,toilettes,sexe_f,acte_naissance,eau_amelioree,animaux,stunt,grappe,menage,id_menage,ligne_enfant
0,33,Maritime,Primaire,Réseau,Feu ouvert,Chasse d'eau,1,1.0,0.0,0,0,1,7,1-7,5
1,41,Maritime,Primaire,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,0,1,15,1-15,2
2,12,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,1,0.0,1.0,0,0,1,18,1-18,6
3,22,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,0,1.0,1.0,0,0,2,1,2-1,4
4,3,Maritime,Secondaire ou plus,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,0,2,2,2-2,2


# Features / label

In [9]:
df.columns

Index(['age_mois', 'region', 'edu_mere', 'electricite', 'cuisiniere',
       'toilettes', 'sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux',
       'stunt', 'grappe', 'menage', 'id_menage', 'ligne_enfant'],
      dtype='object')

In [10]:
NON_ENTREES = ["grappe", "menage", "id_menage", "ligne_enfant", 'stunt']

In [11]:
X = df.drop(columns='stunt')# Entrees (features)
y = df['stunt'] # Sortie (label)

In [12]:
print(f"\nVérification : {X.columns.tolist()}")


Vérification : ['age_mois', 'region', 'edu_mere', 'electricite', 'cuisiniere', 'toilettes', 'sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux', 'grappe', 'menage', 'id_menage', 'ligne_enfant']


In [13]:
y.unique()

array([0, 1], dtype=int8)

In [14]:
X.head()

,age_mois,region,edu_mere,electricite,cuisiniere,toilettes,sexe_f,acte_naissance,eau_amelioree,animaux,grappe,menage,id_menage,ligne_enfant
0,33,Maritime,Primaire,Réseau,Feu ouvert,Chasse d'eau,1,1.0,0.0,0,1,7,1-7,5
1,41,Maritime,Primaire,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,1,15,1-15,2
2,12,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,1,0.0,1.0,0,1,18,1-18,6
3,22,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,0,1.0,1.0,0,2,1,2-1,4
4,3,Maritime,Secondaire ou plus,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,2,2,2-2,2


# Train_test_split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=RANDOM_STATE,
    test_size=0.2,
    stratify=y
)

# Repartition 

print(f"Train : {X_train.shape[0]} observations")
print(f"Test  : {X_test.shape[0]} observations")

print("\nProportions des classes (train) :")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nProportions des classes (test) :")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Train : 3891 observations
Test  : 973 observations

Proportions des classes (train) :
stunt
0    0.756104
1    0.243896
Name: proportion, dtype: float64

Proportions des classes (test) :
stunt
0    0.756423
1    0.243577
Name: proportion, dtype: float64


# Preprocessor

In [16]:
print("\n")
print(f"{30*'-'}Colonnes numériques{30*'-'}")
print("\n")
for c in X.select_dtypes(include='number').columns :
    print(f"{c}{10*' - '}{df[c].nunique()}")
    print()

print("\n")

print(f"{30*'-'}Colonnes non-numériques{30*'-'}")
print("\n")
for c in X.select_dtypes(include='object').columns :
    print(f"{c}{10*' - '}{df[c].unique()}{10*' - '}{df[c].nunique()}")
    print()



------------------------------Colonnes numériques------------------------------


age_mois -  -  -  -  -  -  -  -  -  - 60

sexe_f -  -  -  -  -  -  -  -  -  - 2

acte_naissance -  -  -  -  -  -  -  -  -  - 2

eau_amelioree -  -  -  -  -  -  -  -  -  - 2

animaux -  -  -  -  -  -  -  -  -  - 2

grappe -  -  -  -  -  -  -  -  -  - 419

menage -  -  -  -  -  -  -  -  -  - 21

ligne_enfant -  -  -  -  -  -  -  -  -  - 20



------------------------------Colonnes non-numériques------------------------------


region -  -  -  -  -  -  -  -  -  - ['Maritime' 'Plateaux' 'Centrale' 'Kara' 'Savanes' 'Lomé Commune'
 'Golfe Urbain'] -  -  -  -  -  -  -  -  -  - 7

edu_mere -  -  -  -  -  -  -  -  -  - ['Primaire' 'Aucune' 'Secondaire ou plus'] -  -  -  -  -  -  -  -  -  - 3

electricite -  -  -  -  -  -  -  -  -  - ['Réseau' 'Non' 'Hors réseau'] -  -  -  -  -  -  -  -  -  - 3

cuisiniere -  -  -  -  -  -  -  -  -  - ['Feu ouvert' 'Combustible solide (cuisinière)'
 'Électricité, gaz ou liquide' 

In [17]:
X['eau_amelioree'].unique()

array([0., 1.])

In [18]:
nominales = ['region', 'electricite', 'cuisiniere', 'toilettes', 'sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux']